# NDTI vs. Precipitation — Correlation & Plots

This notebook:
- Loads two spreadsheets (CSV or Excel): **precipitation** and **NDTI**.
- Parses dates and normalizes column names.
- Aligns series by date using one of three strategies:
  - **exact**: only same-day matches
  - **nearest**: nearest NDTI within a tolerance window (e.g., `2D`)
  - **resample**: daily resampling; precipitation summed per day, NDTI daily mean with time interpolation
- Plots:
  - Time series: precipitation (bars) + NDTI (line) on twin axes
  - Scatter with an optional linear trend line
- Computes **Pearson** and **Spearman** correlations.
- Saves merged CSV and PNG figures next to the notebook (in your working directory).

> Tip: If you expect a negative relationship, look for **Pearson r < 0**.


## 1) Setup — file paths and options

- Set `precip_path` to your precipitation spreadsheet (CSV/Excel).
- Set `ndti_path` to your NDTI spreadsheet (CSV/Excel).
- Choose `match_mode` = `"exact"`, `"nearest"`, or `"resample"`.
- If using `"nearest"`, set `tolerance` (e.g., `"1D"`, `"2D"`, `"12H"`).
- `outprefix` controls the output filenames.


In [1]:
# === USER INPUTS ===
# Point these to your files (relative or absolute paths).
precip_path = "/Volumes/External/TJ_SAR/analysis/_data/PRISM_ppt_tmean_early_4km_20200101_20250701_32.5433_-117.1165.csv"   # e.g., "data/precip.xlsx" or "precip.csv"
ndti_path   = "/Volumes/External/TJ_Optical/NDTI.csv"     # e.g., "data/ndti.xlsx" or "ndti.csv"

# How to align dates: "exact", "nearest", or "resample"
match_mode = "exact"

# If match_mode == "nearest", how far can an NDTI date be from a precip date?
tolerance = "1D"  # e.g., "2D", "12H"

# Output filename prefix (without extension)
outprefix = "ndti_precip"


## 2) Imports

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Ensure plots render in notebook
# (Uncomment next line if running in classic Jupyter)
# %matplotlib inline


## 3) Helper functions

In [8]:
def read_table(path: str) -> pd.DataFrame:
    p = Path(path)
    ext = p.suffix.lower()
    if ext in [".xlsx", ".xls"]:
        df = pd.read_excel(p)
    elif ext in [".csv", ".txt"]:
        try:
            df = pd.read_csv(p)
        except Exception:
            df = pd.read_csv(p, sep="\t")
    else:
        raise ValueError(f"Unsupported file extension: {ext}")
    # Normalize column names
    df.columns = [c.strip() for c in df.columns]
    return df

def coerce_date_column(df: pd.DataFrame) -> pd.Series:
    # Prefer exact 'date' match (case-insensitive), fallback to any containing 'date'
    candidates = [c for c in df.columns if c.strip().lower() == "date"]
    if not candidates:
        candidates = [c for c in df.columns if "date" in c.strip().lower()]
    if not candidates:
        raise ValueError("No date column found. Expected a column like 'Date'.")
    col = candidates[0]
    s = pd.to_datetime(df[col], errors="coerce", format="%m/%d/%y")
    if s.isna().all():
        s = pd.to_datetime(df[col], errors="coerce", format="%m/%d/%y")
    if s.isna().all():
        raise ValueError(f"Could not parse any dates from column '{col}'.")
    return s

def pick_numeric_column(df: pd.DataFrame, prefer_contains: str | None = None) -> str:
    # Prefer a numeric column containing a keyword (e.g., 'ndti')
    if prefer_contains:
        cands = [c for c in df.columns if prefer_contains.lower() in c.lower()]
        for c in cands:
            if pd.api.types.is_numeric_dtype(df[c]):
                return c
    # Else first numeric column
    for c in df.columns:
        if c.strip().lower() == "date":
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            return c
    # Try to coerce to numeric
    for c in df.columns:
        if c.strip().lower() == "date":
            continue
        coerced = pd.to_numeric(df[c], errors="coerce")
        if not coerced.isna().all():
            df[c] = coerced
            return c
    raise ValueError("No numeric data column found.")

def aggregate_by_date(df: pd.DataFrame, date_col: str, value_col: str, how: str) -> pd.DataFrame:
    out = df[[date_col, value_col]].copy()
    out = out.dropna(subset=[date_col])
    out[value_col] = pd.to_numeric(out[value_col], errors="coerce")
    out = out.dropna(subset=[value_col])
    if how == "sum":
        out = out.groupby(date_col, as_index=False)[value_col].sum()
    elif how == "mean":
        out = out.groupby(date_col, as_index=False)[value_col].mean()
    else:
        raise ValueError("how must be 'sum' or 'mean'")
    return out

def merge_data(precip: pd.DataFrame, ndti: pd.DataFrame, match: str, tolerance: str | None):
    precip = precip.sort_values("date")
    ndti   = ndti.sort_values("date")
    if match == "exact":
        merged = pd.merge(precip, ndti, on="date", how="inner", suffixes=("_precip", "_ndti"))
        return merged
    elif match == "nearest":
        tol = pd.Timedelta(tolerance) if tolerance else pd.Timedelta("1D")
        merged = pd.merge_asof(precip, ndti, on="date", direction="nearest", tolerance=tol, suffixes=("_precip", "_ndti"))
        merged = merged.dropna(subset=[merged.columns[-1]])
        return merged
    elif match == "resample":
        precip_rs = precip.set_index("date").resample("D").sum().rename(columns={precip.columns[1]: "precip"})
        ndti_rs   = ndti.set_index("date").resample("D").mean().rename(columns={ndti.columns[1]: "ndti"})
        ndti_rs["ndti"] = ndti_rs["ndti"].interpolate(method="time", limit_direction="both")
        merged = precip_rs.join(ndti_rs, how="inner").reset_index().rename(columns={"index": "date"})
        merged = merged.rename(columns={"precip": "precip_amount", "ndti": "mean_ndti"})
        return merged
    else:
        raise ValueError("match must be one of: exact, nearest, resample")

def compute_correlations(df: pd.DataFrame, xcol: str, ycol: str):
    sub = df[[xcol, ycol]].dropna()
    pearson = sub[xcol].corr(sub[ycol], method="pearson")
    spearman = sub[xcol].corr(sub[ycol], method="spearman")
    return pearson, spearman, len(sub)

def plot_timeseries(df: pd.DataFrame, date_col: str, precip_col: str, ndti_col: str, outpath: Path | None = None):
    plt.figure(figsize=(11, 6))
    ax1 = plt.gca()
    ax2 = ax1.twinx()
    ax1.bar(df[date_col], df[precip_col], width=1.0, align="center")
    ax1.set_ylabel("Precipitation")
    ax1.set_xlabel("Date")
    ax2.plot(df[date_col], df[ndti_col], linewidth=2)
    ax2.set_ylabel("Mean NDTI")
    ax1.set_title("Precipitation vs. NDTI (Time Series)")
    plt.tight_layout()
    if outpath is not None:
        plt.savefig(outpath, dpi=200)
    plt.show()

def plot_scatter(df: pd.DataFrame, precip_col: str, ndti_col: str, outpath: Path | None = None):
    x = df[precip_col].astype(float)
    y = df[ndti_col].astype(float)
    plt.figure(figsize=(7, 6))
    plt.scatter(x, y, alpha=0.7)
    if len(x.dropna()) >= 2:
        coeffs = np.polyfit(x, y, 1)
        xx = np.linspace(x.min(), x.max(), 100)
        yy = coeffs[0] * xx + coeffs[1]
        plt.plot(xx, yy, linewidth=2)
    plt.xlabel("Precipitation")
    plt.ylabel("Mean NDTI")
    plt.title("NDTI vs. Precipitation (Scatter)")
    plt.tight_layout()
    if outpath is not None:
        plt.savefig(outpath, dpi=200)
    plt.show()

## 4) Load data

In [9]:
precip_df = read_table(precip_path)
ndti_df   = read_table(ndti_path)

# Parse dates into a standard 'date' column
precip_df["date"] = coerce_date_column(precip_df)
ndti_df["date"]   = coerce_date_column(ndti_df)

# Pick value columns
precip_val_col = None
for c in precip_df.columns:
    if c.strip().lower() != "date" and pd.api.types.is_numeric_dtype(precip_df[c]):
        precip_val_col = c
        break
if precip_val_col is None:
    # Try coercion
    for c in precip_df.columns:
        if c.strip().lower() == "date":
            continue
        precip_df[c] = pd.to_numeric(precip_df[c], errors="coerce")
        if not precip_df[c].isna().all():
            precip_val_col = c
            break
if precip_val_col is None:
    raise ValueError("Could not find a numeric precipitation column.")

ndti_val_col = None
# Prefer column containing 'ndti'
for c in ndti_df.columns:
    if c.strip().lower() != "date" and "ndti" in c.strip().lower() and pd.api.types.is_numeric_dtype(ndti_df[c]):
        ndti_val_col = c
        break
if ndti_val_col is None:
    for c in ndti_df.columns:
        if c.strip().lower() == "date":
            continue
        ndti_df[c] = pd.to_numeric(ndti_df[c], errors="coerce")
        if not ndti_df[c].isna().all():
            ndti_val_col = c
            break
if ndti_val_col is None:
    raise ValueError("Could not find a numeric NDTI column.")
    
# Keep only necessary columns with normalized names
precip_df = precip_df[["date", precip_val_col]].rename(columns={precip_val_col: "precip_amount"})
ndti_df   = ndti_df[["date", ndti_val_col]].rename(columns={ndti_val_col: "mean_ndti"})

# Aggregate duplicates by date
precip_daily = aggregate_by_date(precip_df, "date", "precip_amount", how="sum")
ndti_daily   = aggregate_by_date(ndti_df, "date", "mean_ndti", how="mean")

display(precip_daily.head())
display(ndti_daily.head())

,date,precip_amount
0,2020-01-01,0.0
1,2020-01-02,0.0
2,2020-01-03,0.0
3,2020-01-04,0.0
4,2020-01-05,0.0


,date,mean_ndti


## 5) Merge on date

In [10]:
merged = merge_data(precip_daily, ndti_daily, match=match_mode, tolerance=tolerance)

if merged.empty:
    raise SystemExit("No overlapping dates after merge. Try match_mode='nearest' or 'resample' with a larger window.")

# Identify column names (robust to different merge modes)
precip_col = [c for c in merged.columns if "precip" in c.lower()][0]
ndti_col   = [c for c in merged.columns if "ndti" in c.lower()][0]

merged = merged.sort_values("date").reset_index(drop=True)
merged.head()

SystemExit: No overlapping dates after merge. Try match_mode='nearest' or 'resample' with a larger window.

/Users/ereilly/RS/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3680: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 6) Correlations

In [ ]:
pearson, spearman, n = compute_correlations(merged, precip_col, ndti_col)
trend = "negative" if pearson < 0 else "positive" if pearson > 0 else "zero"

print(f"Merged rows used for correlation: {n}")
print(f"Pearson r: {pearson:.3f} ({trend})")
print(f"Spearman ρ: {spearman:.3f}")

## 7) Plots

In [ ]:
# Time series
ts_path = Path(f"{outprefix}.timeseries.png")
plot_timeseries(merged, "date", precip_col, ndti_col, outpath=ts_path)

In [ ]:
# Scatter with linear fit
sc_path = Path(f"{outprefix}.scatter.png")
plot_scatter(merged, precip_col, ndti_col, outpath=sc_path)

## 8) Save merged data

In [ ]:
merged_path = Path(f"{outprefix}.merged.csv")
merged.to_csv(merged_path, index=False)
print(f"Saved merged data to: {merged_path}")

---

*Notebook generated on 2025-08-13 20:02:56 UTC.*